In [1]:
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple, List, Union

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, brier_score_loss, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import PartialDependenceDisplay

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import shap

# Counterfactuals
import dice_ml
from dice_ml import Dice


# -------------------------
# Utility metrics
# -------------------------
def safe_mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100.0

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)), eps)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

def decile_report(y_true, y_score, n_bins=10) -> pd.DataFrame:
    """
    Works for:
      - Classification: y_score = predicted probability for positive class
      - Regression: y_score = predicted value (useful for decile slicing)
    """
    df = pd.DataFrame({"y_true": y_true, "y_score": y_score}).dropna()
    df["decile"] = pd.qcut(df["y_score"].rank(method="first"), q=n_bins, labels=False) + 1
    grp = df.groupby("decile", as_index=False).agg(
        count=("y_true", "size"),
        y_true_mean=("y_true", "mean"),
        y_score_mean=("y_score", "mean"),
        y_true_min=("y_true", "min"),
        y_true_max=("y_true", "max"),
        y_score_min=("y_score", "min"),
        y_score_max=("y_score", "max"),
    )
    grp["decile"] = grp["decile"].astype(int)
    return grp.sort_values("decile")

def ks_table_binary(y_true, y_prob, n_bins=10) -> pd.DataFrame:
    """
    KS decile table for binary classification.
    """
    df = pd.DataFrame({"y": y_true, "p": y_prob}).dropna()
    df = df.sort_values("p", ascending=False).reset_index(drop=True)
    df["decile"] = pd.qcut(df.index + 1, q=n_bins, labels=False) + 1

    out = []
    total_pos = (df["y"] == 1).sum()
    total_neg = (df["y"] == 0).sum()

    cum_pos = 0
    cum_neg = 0

    for d in range(1, n_bins + 1):
        sub = df[df["decile"] == d]
        pos = (sub["y"] == 1).sum()
        neg = (sub["y"] == 0).sum()
        cum_pos += pos
        cum_neg += neg

        tpr = cum_pos / max(total_pos, 1)
        fpr = cum_neg / max(total_neg, 1)
        ks = abs(tpr - fpr)

        out.append({
            "decile": d,
            "count": len(sub),
            "pos": pos,
            "neg": neg,
            "cum_pos": cum_pos,
            "cum_neg": cum_neg,
            "tpr(cum_pos/total_pos)": tpr,
            "fpr(cum_neg/total_neg)": fpr,
            "ks": ks,
            "p_min": sub["p"].min(),
            "p_max": sub["p"].max(),
            "p_mean": sub["p"].mean(),
        })

    ks_df = pd.DataFrame(out)
    ks_df["decile"] = ks_df["decile"].astype(int)
    return ks_df

def psi(expected: pd.Series, actual: pd.Series, buckets=10, eps=1e-6) -> float:
    """
    Population Stability Index for a single numeric feature.
    """
    expected = expected.dropna()
    actual = actual.dropna()

    quantiles = np.quantile(expected, np.linspace(0, 1, buckets + 1))
    quantiles = np.unique(quantiles)
    if len(quantiles) < 3:  # too few unique values
        return np.nan

    exp_counts, _ = np.histogram(expected, bins=quantiles)
    act_counts, _ = np.histogram(actual, bins=quantiles)

    exp_perc = exp_counts / max(exp_counts.sum(), 1)
    act_perc = act_counts / max(act_counts.sum(), 1)

    exp_perc = np.clip(exp_perc, eps, 1)
    act_perc = np.clip(act_perc, eps, 1)

    return float(np.sum((act_perc - exp_perc) * np.log(act_perc / exp_perc)))


# -------------------------
# Plot helpers (Plotly = “fancy”)
# -------------------------
def plot_roc(y_true, y_prob, title="ROC Curve"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name="ROC"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random", line=dict(dash="dash")))
    fig.update_layout(title=title, xaxis_title="False Positive Rate", yaxis_title="True Positive Rate")
    fig.show()

def plot_pr(y_true, y_prob, title="Precision-Recall Curve"):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines", name="PR"))
    fig.update_layout(title=title, xaxis_title="Recall", yaxis_title="Precision")
    fig.show()

def plot_calibration(y_true, y_prob, n_bins=10, title="Calibration Curve"):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="quantile")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=prob_pred, y=prob_true, mode="lines+markers", name="Model"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect", line=dict(dash="dash")))
    fig.update_layout(title=title, xaxis_title="Predicted probability", yaxis_title="Observed frequency")
    fig.show()

def plot_gain_lift(y_true, y_prob, n_bins=10, title="Cumulative Gain"):
    df = pd.DataFrame({"y": y_true, "p": y_prob}).sort_values("p", ascending=False).reset_index(drop=True)
    df["bucket"] = pd.qcut(df.index + 1, q=n_bins, labels=False) + 1
    df["cum_pos"] = (df["y"] == 1).cumsum()
    total_pos = (df["y"] == 1).sum()
    df["cum_gain"] = df["cum_pos"] / max(total_pos, 1)
    df["population"] = (df.index + 1) / len(df)

    # Take bucket endpoints for cleaner plot
    endpoints = df.groupby("bucket").tail(1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=endpoints["population"], y=endpoints["cum_gain"], mode="lines+markers", name="Model"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random", line=dict(dash="dash")))
    fig.update_layout(title=title, xaxis_title="Population fraction", yaxis_title="Cumulative gain (TP fraction)")
    fig.show()

def plot_residuals(y_true, y_pred, title="Residual Plot"):
    resid = y_true - y_pred
    fig = px.scatter(x=y_pred, y=resid, labels={"x": "Predicted", "y": "Residual"}, title=title)
    fig.add_hline(y=0, line_dash="dash")
    fig.show()

def plot_actual_vs_pred(y_true, y_pred, title="Actual vs Predicted"):
    fig = px.scatter(x=y_true, y=y_pred, labels={"x": "Actual", "y": "Predicted"}, title=title)
    # 45-degree line
    mn = min(np.min(y_true), np.min(y_pred))
    mx = max(np.max(y_true), np.max(y_pred))
    fig.add_trace(go.Scatter(x=[mn, mx], y=[mn, mx], mode="lines", name="Ideal", line=dict(dash="dash")))
    fig.show()

def plot_decile_table(decile_df: pd.DataFrame, title="Decile Report"):
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(decile_df.columns)),
        cells=dict(values=[decile_df[c] for c in decile_df.columns])
    )])
    fig.update_layout(title=title)
    fig.show()


# -------------------------
# Main FidxAI-style runner
# -------------------------
@dataclass
class FidxLiteConfig:
    problem_type: str  # "classification" or "regression"
    positive_class: int = 1  # only for binary classification
    shap_sample: int = 2000  # sample rows for SHAP for speed
    deciles: int = 10

class FidxLite:
    def __init__(self, config: FidxLiteConfig):
        self.cfg = config
        self.model = None
        self.feature_names = None
        self.train_df = None  # for drift & counterfactual data schema
        self.target_name = None

    def fit(self, model, X_train: Union[pd.DataFrame, np.ndarray], y_train: Union[pd.Series, np.ndarray],
            train_df: Optional[pd.DataFrame] = None, target_name: str = "target"):
        self.model = model
        self.target_name = target_name

        if isinstance(X_train, pd.DataFrame):
            self.feature_names = list(X_train.columns)
        else:
            self.feature_names = [f"f{i}" for i in range(X_train.shape[1])]

        # Keep a training dataframe for DiCE and drift tooling
        if train_df is not None:
            self.train_df = train_df.copy()
        else:
            # Build one if not provided
            Xdf = pd.DataFrame(X_train, columns=self.feature_names)
            self.train_df = Xdf.copy()
            self.train_df[target_name] = y_train

        return self

    # -------------------------
    # 1) Evaluation Reports
    # -------------------------
    def evaluate(self, X_test, y_test) -> Dict[str, Any]:
        if self.cfg.problem_type == "regression":
            y_pred = self.model.predict(X_test)
            metrics = {
                "RMSE": float(np.sqrt(mean_squared_error(y_test, y_pred))),
                "MSE": float(mean_squared_error(y_test, y_pred)),
                "MAE": float(mean_absolute_error(y_test, y_pred)),
                "MAPE(%)": float(safe_mape(y_test, y_pred)),
                "SMAPE(%)": float(smape(y_test, y_pred)),
                "R2": float(r2_score(y_test, y_pred)),
            }
            dec = decile_report(y_test, y_pred, n_bins=self.cfg.deciles)

            print("=== Regression Metrics ===")
            for k, v in metrics.items():
                print(f"{k}: {v:.5f}")

            plot_actual_vs_pred(np.asarray(y_test), np.asarray(y_pred))
            plot_residuals(np.asarray(y_test), np.asarray(y_pred))
            plot_decile_table(dec, title="Regression Decile Report (by Predicted Value)")

            return {"metrics": metrics, "decile_report": dec}

        # Classification
        # Determine if binary vs multi-class
        y_pred = self.model.predict(X_test)

        # Probabilities if available
        y_proba = None
        if hasattr(self.model, "predict_proba"):
            y_proba = self.model.predict_proba(X_test)

        # If binary, focus on positive-class probability
        unique_classes = np.unique(y_test)
        is_binary = (len(unique_classes) == 2)

        print("=== Classification Report ===")
        # (This avoids your earlier “mix of binary and continuous” error by using class labels, not probabilities)
        print(classification_report(y_test, y_pred))

        metrics = {
            "Accuracy": float(accuracy_score(y_test, y_pred)),
            "MCC": float(matthews_corrcoef(y_test, y_pred)),
        }

        if is_binary and (y_proba is not None):
            p = y_proba[:, 1]
            metrics.update({
                "Precision": float(precision_score(y_test, y_pred)),
                "Recall": float(recall_score(y_test, y_pred)),
                "F1": float(f1_score(y_test, y_pred)),
                "AUROC": float(roc_auc_score(y_test, p)),
                "LogLoss": float(log_loss(y_test, y_proba)),
                "Brier": float(brier_score_loss(y_test, p)),
            })

            print("=== Binary Classification Metrics ===")
            for k, v in metrics.items():
                print(f"{k}: {v:.5f}")

            plot_roc(y_test, p)
            plot_pr(y_test, p)
            plot_calibration(y_test, p, n_bins=self.cfg.deciles)
            plot_gain_lift(y_test, p, n_bins=self.cfg.deciles)

            ks_df = ks_table_binary(y_test, p, n_bins=self.cfg.deciles)
            plot_decile_table(ks_df, title="KS Decile Report (Binary)")

            return {"metrics": metrics, "ks_decile": ks_df}

        # Multi-class metrics (limited “fancy plots” because ROC/KS are more involved)
        metrics.update({
            "MacroF1": float(f1_score(y_test, y_pred, average="macro")),
            "WeightedF1": float(f1_score(y_test, y_pred, average="weighted")),
            "MacroPrecision": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
            "MacroRecall": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
        })
        print("=== Multi-Class Metrics ===")
        for k, v in metrics.items():
            print(f"{k}: {v:.5f}")

        cm = confusion_matrix(y_test, y_pred)
        fig = px.imshow(cm, text_auto=True, title="Confusion Matrix", labels=dict(x="Predicted", y="Actual"))
        fig.show()

        return {"metrics": metrics, "confusion_matrix": cm}

    # -------------------------
    # 2) Dynamic Threshold (Binary classification)
    # -------------------------
    def threshold_sweep(self, X_val, y_val, thresholds=None) -> pd.DataFrame:
        if self.cfg.problem_type != "classification":
            raise ValueError("threshold_sweep is only for classification.")
        if not hasattr(self.model, "predict_proba"):
            raise ValueError("Model must support predict_proba for threshold tuning.")

        thresholds = thresholds or np.linspace(0.05, 0.95, 19)
        proba = self.model.predict_proba(X_val)[:, 1]

        rows = []
        for t in thresholds:
            pred = (proba >= t).astype(int)
            rows.append({
                "threshold": float(t),
                "precision": float(precision_score(y_val, pred, zero_division=0)),
                "recall": float(recall_score(y_val, pred, zero_division=0)),
                "f1": float(f1_score(y_val, pred, zero_division=0)),
                "accuracy": float(accuracy_score(y_val, pred)),
                "mcc": float(matthews_corrcoef(y_val, pred)),
            })

        df = pd.DataFrame(rows)
        fig = px.line(df, x="threshold", y=["precision", "recall", "f1", "accuracy", "mcc"],
                      title="Dynamic Threshold Sweep (Binary)")
        fig.show()
        return df

    # -------------------------
    # 3) Interpretability (SHAP + PDP)
    # -------------------------
    def shap_global(self, X: pd.DataFrame):
        Xdf = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.feature_names)
        Xs = Xdf.sample(min(len(Xdf), self.cfg.shap_sample), random_state=42)

        # Auto-choose explainer
        if hasattr(self.model, "predict_proba") and self.cfg.problem_type == "classification":
            f = lambda data: self.model.predict_proba(pd.DataFrame(data, columns=self.feature_names))
            explainer = shap.Explainer(f, Xs)
            shap_values = explainer(Xs)
            # For binary, take class 1 explanations
            if shap_values.values.ndim == 3:
                shap.summary_plot(shap_values.values[:, :, 1], Xs, show=True)
            else:
                shap.summary_plot(shap_values, Xs, show=True)
        else:
            explainer = shap.Explainer(self.model.predict, Xs)
            shap_values = explainer(Xs)
            shap.summary_plot(shap_values, Xs, show=True)

        return shap_values

    def shap_local(self, X_row: pd.DataFrame):
        Xdf = X_row if isinstance(X_row, pd.DataFrame) else pd.DataFrame([X_row], columns=self.feature_names)

        if hasattr(self.model, "predict_proba") and self.cfg.problem_type == "classification":
            f = lambda data: self.model.predict_proba(pd.DataFrame(data, columns=self.feature_names))
            explainer = shap.Explainer(f, Xdf)
            sv = explainer(Xdf)
            # Show waterfall for class 1 if binary
            if sv.values.ndim == 3:
                shap.plots.waterfall(shap.Explanation(values=sv.values[0, :, 1],
                                                      base_values=sv.base_values[0, 1],
                                                      data=Xdf.iloc[0],
                                                      feature_names=self.feature_names))
            else:
                shap.plots.waterfall(sv[0])
        else:
            explainer = shap.Explainer(self.model.predict, Xdf)
            sv = explainer(Xdf)
            shap.plots.waterfall(sv[0])

        return sv

    def pdp(self, X: pd.DataFrame, features: List[Union[int, str]]):
        Xdf = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.feature_names)
        fig, ax = plt.subplots(figsize=(10, 5))
        PartialDependenceDisplay.from_estimator(self.model, Xdf, features=features, ax=ax)
        plt.title("Partial Dependence Plot(s)")
        plt.show()

    # -------------------------
    # 4) Counterfactuals (DiCE)
    # -------------------------
    def counterfactuals(self, query_instances: pd.DataFrame, total_CFs=3,
                        desired_class: str = "opposite", desired_range: Optional[List[float]] = None):
        """
        - For classification: desired_class can be "opposite" or a specific class label as string.
        - For regression: pass desired_range like [lower, upper]
        """
        if self.train_df is None:
            raise ValueError("train_df is required for counterfactuals. Pass it to fit().")

        data = dice_ml.Data(dataframe=self.train_df,
                            continuous_features=[c for c in self.train_df.columns if c != self.target_name and pd.api.types.is_numeric_dtype(self.train_df[c])],
                            outcome_name=self.target_name)

        if self.cfg.problem_type == "classification":
            model = dice_ml.Model(model=self.model, backend="sklearn", model_type="classifier")
        else:
            model = dice_ml.Model(model=self.model, backend="sklearn", model_type="regressor")

        dice = Dice(data, model, method="random")  # you can switch to "genetic"
        cf = dice.generate_counterfactuals(
            query_instances=query_instances,
            total_CFs=total_CFs,
            desired_class=desired_class if self.cfg.problem_type == "classification" else None,
            desired_range=desired_range if self.cfg.problem_type == "regression" else None,
        )
        # Display nicely
        cf.visualize_as_dataframe()
        return cf

    # -------------------------
    # 5) Drift / PSI (simple, useful)
    # -------------------------
    def drift_report(self, X_ref: pd.DataFrame, X_cur: pd.DataFrame, top_n=15) -> pd.DataFrame:
        Xr = X_ref if isinstance(X_ref, pd.DataFrame) else pd.DataFrame(X_ref, columns=self.feature_names)
        Xc = X_cur if isinstance(X_cur, pd.DataFrame) else pd.DataFrame(X_cur, columns=self.feature_names)

        rows = []
        for col in self.feature_names:
            if pd.api.types.is_numeric_dtype(Xr[col]):
                val = psi(Xr[col], Xc[col], buckets=self.cfg.deciles)
                rows.append({"feature": col, "psi": val})

        df = pd.DataFrame(rows).dropna().sort_values("psi", ascending=False)
        display_df = df.head(top_n)

        fig = px.bar(display_df, x="feature", y="psi", title=f"Top {top_n} Features by PSI (Higher = More Drift)")
        fig.show()

        # Optional: show distributions for top few drifted features
        for col in display_df["feature"].head(min(5, len(display_df))):
            tmp = pd.DataFrame({
                col: pd.concat([Xr[col].rename("ref"), Xc[col].rename("cur")], axis=0),
                "dataset": ["ref"] * len(Xr) + ["cur"] * len(Xc)
            })
            fig2 = px.histogram(tmp, x=col, color="dataset", barmode="overlay",
                                title=f"Distribution Drift: {col}")
            fig2.show()

        return df

/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Union

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, brier_score_loss, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import PartialDependenceDisplay

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import shap

# Counterfactuals
import dice_ml
from dice_ml import Dice


# -------------------------
# Utility metrics
# -------------------------
def safe_mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100.0

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)), eps)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0

def _to_df(X, feature_names=None) -> pd.DataFrame:
    if isinstance(X, pd.DataFrame):
        return X.copy()
    X = np.asarray(X)
    if feature_names is None:
        feature_names = [f"f{i}" for i in range(X.shape[1])]
    return pd.DataFrame(X, columns=feature_names)


# -------------------------
# Fidx-style bin reports (Decile / Percentile / any N)
# -------------------------
def _bin_table_binary(y_true, y_prob, X_raw=None, bins=10) -> (pd.DataFrame, pd.DataFrame):
    """
    Fidx-style table for binary classification.
    Sort by predicted prob (desc), split into bins, compute responders + cumulative metrics + lift.
    Optionally append raw feature means (X_raw columns).
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)

    if X_raw is not None:
        X_raw = _to_df(X_raw)
        if len(X_raw) != len(y_true):
            raise ValueError("X_raw rows must match y_true length.")

    df = pd.DataFrame({"pred_prob": y_prob, "y": y_true})
    if X_raw is not None:
        df = pd.concat([df, X_raw.reset_index(drop=True)], axis=1)

    df = df.sort_values("pred_prob", ascending=False).reset_index(drop=True)
    splits = np.array_split(df, bins)

    total_resp = df["y"].sum()
    rand_mean = df["y"].mean()  # baseline response rate

    raw_cols = list(X_raw.columns) if X_raw is not None else []
    rows = []

    cum_resp = 0
    cum_n = 0

    for i, chunk in enumerate(splits, start=1):
        n = len(chunk)
        if n == 0:
            continue

        responders = chunk["y"].sum()
        cum_resp += responders
        cum_n += n

        pred_mean = chunk["pred_prob"].mean()
        actual_mean = chunk["y"].mean()

        cum_precision = cum_resp / max(cum_n, 1)
        cum_recall = cum_resp / max(total_resp, 1)
        lift = cum_precision / max(rand_mean, 1e-12)

        row = {
            "BIN": i,
            "N": int(n),
            "PRED_MEAN": round(float(pred_mean), 6),
            "ACTUAL_MEAN": round(float(actual_mean), 6),
            "RESPONDERS": int(responders),
            "CUM_RESPONDERS": int(cum_resp),
            "CUM_PRECISION": round(float(cum_precision), 6),
            "CUM_RECALL": round(float(cum_recall), 6),
            "LIFT": round(float(lift), 6),
            "PRED_MIN": round(float(chunk["pred_prob"].min()), 6),
            "PRED_MAX": round(float(chunk["pred_prob"].max()), 6),
        }

        for c in raw_cols:
            row[f"{c} MEAN"] = round(float(chunk[c].mean()), 6)

        rows.append(row)

    out = pd.DataFrame(rows)
    if bins == 10:
        out = out.rename(columns={"BIN": "DECILE"})
    elif bins == 100:
        out = out.rename(columns={"BIN": "PERCENTILE"})
    else:
        out = out.rename(columns={"BIN": f"BIN_{bins}"})

    return out, df


def _bin_table_regression(y_true, y_pred, X_raw=None, bins=10) -> (pd.DataFrame, pd.DataFrame):
    """
    Fidx-style table for regression:
    Sort by predicted value (desc), split into bins, compute bin-level error diagnostics.
    Optionally append raw feature means (X_raw columns).
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    if X_raw is not None:
        X_raw = _to_df(X_raw)
        if len(X_raw) != len(y_true):
            raise ValueError("X_raw rows must match y_true length.")

    df = pd.DataFrame({"pred": y_pred, "y": y_true})
    if X_raw is not None:
        df = pd.concat([df, X_raw.reset_index(drop=True)], axis=1)

    df = df.sort_values("pred", ascending=False).reset_index(drop=True)
    splits = np.array_split(df, bins)

    raw_cols = list(X_raw.columns) if X_raw is not None else []
    rows = []

    for i, chunk in enumerate(splits, start=1):
        n = len(chunk)
        if n == 0:
            continue

        yt = chunk["y"].to_numpy()
        yp = chunk["pred"].to_numpy()

        row = {
            "BIN": i,
            "N": int(n),
            "PRED_MEAN": round(float(np.mean(yp)), 6),
            "ACTUAL_MEAN": round(float(np.mean(yt)), 6),
            "MAE": round(float(mean_absolute_error(yt, yp)), 6),
            "RMSE": round(float(np.sqrt(mean_squared_error(yt, yp))), 6),
            "MAPE(%)": round(float(safe_mape(yt, yp)), 6),
            "SMAPE(%)": round(float(smape(yt, yp)), 6),
            "PRED_MIN": round(float(np.min(yp)), 6),
            "PRED_MAX": round(float(np.max(yp)), 6),
        }

        for c in raw_cols:
            row[f"{c} MEAN"] = round(float(chunk[c].mean()), 6)

        rows.append(row)

    out = pd.DataFrame(rows)
    if bins == 10:
        out = out.rename(columns={"BIN": "DECILE"})
    elif bins == 100:
        out = out.rename(columns={"BIN": "PERCENTILE"})
    else:
        out = out.rename(columns={"BIN": f"BIN_{bins}"})

    return out, df


# -------------------------
# KS (kept from your code)
# -------------------------
def ks_table_binary(y_true, y_prob, n_bins=10) -> pd.DataFrame:
    df = pd.DataFrame({"y": y_true, "p": y_prob}).dropna()
    df = df.sort_values("p", ascending=False).reset_index(drop=True)
    df["decile"] = pd.qcut(df.index + 1, q=n_bins, labels=False) + 1

    out = []
    total_pos = (df["y"] == 1).sum()
    total_neg = (df["y"] == 0).sum()
    cum_pos = 0
    cum_neg = 0

    for d in range(1, n_bins + 1):
        sub = df[df["decile"] == d]
        pos = (sub["y"] == 1).sum()
        neg = (sub["y"] == 0).sum()
        cum_pos += pos
        cum_neg += neg

        tpr = cum_pos / max(total_pos, 1)
        fpr = cum_neg / max(total_neg, 1)
        ks = abs(tpr - fpr)

        out.append({
            "decile": d,
            "count": len(sub),
            "pos": int(pos),
            "neg": int(neg),
            "cum_pos": int(cum_pos),
            "cum_neg": int(cum_neg),
            "tpr(cum_pos/total_pos)": float(tpr),
            "fpr(cum_neg/total_neg)": float(fpr),
            "ks": float(ks),
            "p_min": float(sub["p"].min()),
            "p_max": float(sub["p"].max()),
            "p_mean": float(sub["p"].mean()),
        })

    ks_df = pd.DataFrame(out)
    ks_df["decile"] = ks_df["decile"].astype(int)
    return ks_df


# -------------------------
# PSI drift (kept)
# -------------------------
def psi(expected: pd.Series, actual: pd.Series, buckets=10, eps=1e-6) -> float:
    expected = expected.dropna()
    actual = actual.dropna()

    quantiles = np.quantile(expected, np.linspace(0, 1, buckets + 1))
    quantiles = np.unique(quantiles)
    if len(quantiles) < 3:
        return np.nan

    exp_counts, _ = np.histogram(expected, bins=quantiles)
    act_counts, _ = np.histogram(actual, bins=quantiles)

    exp_perc = exp_counts / max(exp_counts.sum(), 1)
    act_perc = act_counts / max(act_counts.sum(), 1)

    exp_perc = np.clip(exp_perc, eps, 1)
    act_perc = np.clip(act_perc, eps, 1)

    return float(np.sum((act_perc - exp_perc) * np.log(act_perc / exp_perc)))


# -------------------------
# Plot helpers (Plotly = “fancy”)
# -------------------------
def plot_roc(y_true, y_prob, title="ROC Curve"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name="ROC"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random", line=dict(dash="dash")))
    fig.update_layout(title=title, xaxis_title="False Positive Rate", yaxis_title="True Positive Rate")
    fig.show()

def plot_pr(y_true, y_prob, title="Precision-Recall Curve"):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines", name="PR"))
    fig.update_layout(title=title, xaxis_title="Recall", yaxis_title="Precision")
    fig.show()

def plot_calibration(y_true, y_prob, n_bins=10, title="Calibration Curve"):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="quantile")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=prob_pred, y=prob_true, mode="lines+markers", name="Model"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect", line=dict(dash="dash")))
    fig.update_layout(title=title, xaxis_title="Predicted probability", yaxis_title="Observed frequency")
    fig.show()

def plot_gain_lift(y_true, y_prob, n_bins=10, title="Cumulative Gain"):
    df = pd.DataFrame({"y": y_true, "p": y_prob}).sort_values("p", ascending=False).reset_index(drop=True)
    df["bucket"] = pd.qcut(df.index + 1, q=n_bins, labels=False) + 1
    df["cum_pos"] = (df["y"] == 1).cumsum()
    total_pos = (df["y"] == 1).sum()
    df["cum_gain"] = df["cum_pos"] / max(total_pos, 1)
    df["population"] = (df.index + 1) / len(df)
    endpoints = df.groupby("bucket").tail(1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=endpoints["population"], y=endpoints["cum_gain"], mode="lines+markers", name="Model"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random", line=dict(dash="dash")))
    fig.update_layout(title=title, xaxis_title="Population fraction", yaxis_title="Cumulative gain (TP fraction)")
    fig.show()

def plot_residuals(y_true, y_pred, title="Residual Plot"):
    resid = y_true - y_pred
    fig = px.scatter(x=y_pred, y=resid, labels={"x": "Predicted", "y": "Residual"}, title=title)
    fig.add_hline(y=0, line_dash="dash")
    fig.show()

def plot_actual_vs_pred(y_true, y_pred, title="Actual vs Predicted"):
    fig = px.scatter(x=y_true, y=y_pred, labels={"x": "Actual", "y": "Predicted"}, title=title)
    mn = min(np.min(y_true), np.min(y_pred))
    mx = max(np.max(y_true), np.max(y_pred))
    fig.add_trace(go.Scatter(x=[mn, mx], y=[mn, mx], mode="lines", name="Ideal", line=dict(dash="dash")))
    fig.show()

def plot_table(df: pd.DataFrame, title="Report Table"):
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(df.columns)),
        cells=dict(values=[df[c] for c in df.columns])
    )])
    fig.update_layout(title=title)
    fig.show()


# -------------------------
# Main FidxAI-style runner
# -------------------------
@dataclass
class FidxLiteConfig:
    problem_type: str  # "classification" or "regression"
    positive_class: int = 1
    shap_sample: int = 2000
    bins: int = 10  # 10=decile, 100=percentile, any N


class FidxLite:
    def __init__(self, config: FidxLiteConfig):
        self.cfg = config
        self.model = None
        self.feature_names = None
        self.train_df = None
        self.target_name = None

    def fit(self, model, X_train: Union[pd.DataFrame, np.ndarray], y_train: Union[pd.Series, np.ndarray],
            train_df: Optional[pd.DataFrame] = None, target_name: str = "target"):
        self.model = model
        self.target_name = target_name

        Xdf = _to_df(X_train)
        self.feature_names = list(Xdf.columns)

        if train_df is not None:
            self.train_df = train_df.copy()
        else:
            self.train_df = Xdf.copy()
            self.train_df[target_name] = np.asarray(y_train).reshape(-1)

        return self

    # -------------------------
    # 1) Evaluation Reports
    # -------------------------
    def evaluate(
        self,
        X_test,
        y_test,
        X_train: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        y_train: Optional[Union[pd.Series, np.ndarray]] = None,
        X_train_raw: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        X_test_raw: Optional[Union[pd.DataFrame, np.ndarray]] = None,
        report_on: str = "test",  # "train" | "test" | "both"
        show_tables: bool = True
    ) -> Dict[str, Any]:

        Xte = _to_df(X_test, feature_names=self.feature_names)
        yte = np.asarray(y_test).reshape(-1)

        out: Dict[str, Any] = {}

        if self.cfg.problem_type == "regression":
            y_pred = np.asarray(self.model.predict(Xte)).reshape(-1)

            metrics = {
                "RMSE": float(np.sqrt(mean_squared_error(yte, y_pred))),
                "MSE": float(mean_squared_error(yte, y_pred)),
                "MAE": float(mean_absolute_error(yte, y_pred)),
                "MAPE(%)": float(safe_mape(yte, y_pred)),
                "SMAPE(%)": float(smape(yte, y_pred)),
                "R2": float(r2_score(yte, y_pred)),
            }

            print("=== Regression Metrics ===")
            for k, v in metrics.items():
                print(f"{k}: {v:.5f}")

            plot_actual_vs_pred(yte, y_pred)
            plot_residuals(yte, y_pred)

            # Bin report (test)
            if report_on in ("test", "both"):
                test_bins, _ = _bin_table_regression(
                    y_true=yte,
                    y_pred=y_pred,
                    X_raw=X_test_raw,
                    bins=self.cfg.bins
                )
                out["test_bin_report"] = test_bins
                if show_tables:
                    plot_table(test_bins, title=f"Regression {'Decile' if self.cfg.bins==10 else 'Percentile' if self.cfg.bins==100 else 'Bin'} Report (TEST)")

            # Bin report (train) if requested
            if report_on in ("train", "both"):
                if X_train is None or y_train is None:
                    raise ValueError("X_train and y_train must be provided when report_on includes 'train'.")
                Xtr = _to_df(X_train, feature_names=self.feature_names)
                ytr = np.asarray(y_train).reshape(-1)
                y_pred_tr = np.asarray(self.model.predict(Xtr)).reshape(-1)

                train_bins, _ = _bin_table_regression(
                    y_true=ytr,
                    y_pred=y_pred_tr,
                    X_raw=X_train_raw,
                    bins=self.cfg.bins
                )
                out["train_bin_report"] = train_bins
                if show_tables:
                    plot_table(train_bins, title=f"Regression {'Decile' if self.cfg.bins==10 else 'Percentile' if self.cfg.bins==100 else 'Bin'} Report (TRAIN)")

            out["metrics"] = metrics
            return out

        # -------------------------
        # Classification
        # -------------------------
        y_pred = self.model.predict(Xte)

        print("=== Classification Report ===")
        print(classification_report(yte, y_pred))

        metrics = {
            "Accuracy": float(accuracy_score(yte, y_pred)),
            "MCC": float(matthews_corrcoef(yte, y_pred)),
        }

        # Detect binary vs multi-class
        unique_classes = np.unique(yte)
        is_binary = (len(unique_classes) == 2)

        # Probabilities (for binary fancy plots + decile tables)
        y_proba = None
        if hasattr(self.model, "predict_proba"):
            y_proba = self.model.predict_proba(Xte)

        if is_binary and y_proba is not None:
            p = y_proba[:, 1]
            metrics.update({
                "Precision": float(precision_score(yte, y_pred, zero_division=0)),
                "Recall": float(recall_score(yte, y_pred, zero_division=0)),
                "F1": float(f1_score(yte, y_pred, zero_division=0)),
                "AUROC": float(roc_auc_score(yte, p)),
                "LogLoss": float(log_loss(yte, y_proba)),
                "Brier": float(brier_score_loss(yte, p)),
            })

            print("=== Binary Classification Metrics ===")
            for k, v in metrics.items():
                print(f"{k}: {v:.5f}")

            plot_roc(yte, p)
            plot_pr(yte, p)
            plot_calibration(yte, p, n_bins=min(self.cfg.bins, 10), title="Calibration Curve")
            plot_gain_lift(yte, p, n_bins=min(self.cfg.bins, 10), title="Cumulative Gain")

            # KS decile report (fixed at deciles concept; uses n_bins=min(bins,10) for sanity)
            ks_df = ks_table_binary(yte, p, n_bins=min(self.cfg.bins, 10))
            out["ks_decile"] = ks_df
            if show_tables:
                plot_table(ks_df, title="KS Report (Binary)")

            # Fidx-style decile/percentile report (TEST)
            if report_on in ("test", "both"):
                test_bins, _ = _bin_table_binary(
                    y_true=yte,
                    y_prob=p,
                    X_raw=X_test_raw,
                    bins=self.cfg.bins
                )
                out["test_bin_report"] = test_bins
                if show_tables:
                    plot_table(test_bins, title=f"Binary Classification {'Decile' if self.cfg.bins==10 else 'Percentile' if self.cfg.bins==100 else 'Bin'} Report (TEST)")

            # Fidx-style decile/percentile report (TRAIN)
            if report_on in ("train", "both"):
                if X_train is None or y_train is None:
                    raise ValueError("X_train and y_train must be provided when report_on includes 'train'.")
                Xtr = _to_df(X_train, feature_names=self.feature_names)
                ytr = np.asarray(y_train).reshape(-1)
                p_tr = self.model.predict_proba(Xtr)[:, 1]

                train_bins, _ = _bin_table_binary(
                    y_true=ytr,
                    y_prob=p_tr,
                    X_raw=X_train_raw,
                    bins=self.cfg.bins
                )
                out["train_bin_report"] = train_bins
                if show_tables:
                    plot_table(train_bins, title=f"Binary Classification {'Decile' if self.cfg.bins==10 else 'Percentile' if self.cfg.bins==100 else 'Bin'} Report (TRAIN)")

            out["metrics"] = metrics
            return out

        # Multi-class branch
        metrics.update({
            "MacroF1": float(f1_score(yte, y_pred, average="macro")),
            "WeightedF1": float(f1_score(yte, y_pred, average="weighted")),
            "MacroPrecision": float(precision_score(yte, y_pred, average="macro", zero_division=0)),
            "MacroRecall": float(recall_score(yte, y_pred, average="macro", zero_division=0)),
        })

        print("=== Multi-Class Metrics ===")
        for k, v in metrics.items():
            print(f"{k}: {v:.5f}")

        cm = confusion_matrix(yte, y_pred)
        fig = px.imshow(cm, text_auto=True, title="Confusion Matrix", labels=dict(x="Predicted", y="Actual"))
        fig.show()

        out["metrics"] = metrics
        out["confusion_matrix"] = cm
        return out

    # -------------------------
    # 2) Dynamic Threshold (Binary classification)
    # -------------------------
    def threshold_sweep(self, X_val, y_val, thresholds=None) -> pd.DataFrame:
        if self.cfg.problem_type != "classification":
            raise ValueError("threshold_sweep is only for classification.")
        if not hasattr(self.model, "predict_proba"):
            raise ValueError("Model must support predict_proba for threshold tuning.")

        Xv = _to_df(X_val, feature_names=self.feature_names)
        yv = np.asarray(y_val).reshape(-1)

        thresholds = thresholds or np.linspace(0.05, 0.95, 19)
        proba = self.model.predict_proba(Xv)[:, 1]

        rows = []
        for t in thresholds:
            pred = (proba >= t).astype(int)
            rows.append({
                "threshold": float(t),
                "precision": float(precision_score(yv, pred, zero_division=0)),
                "recall": float(recall_score(yv, pred, zero_division=0)),
                "f1": float(f1_score(yv, pred, zero_division=0)),
                "accuracy": float(accuracy_score(yv, pred)),
                "mcc": float(matthews_corrcoef(yv, pred)),
            })

        df = pd.DataFrame(rows)
        fig = px.line(df, x="threshold", y=["precision", "recall", "f1", "accuracy", "mcc"],
                      title="Dynamic Threshold Sweep (Binary)")
        fig.show()
        return df

    # -------------------------
    # 3) Interpretability (SHAP + PDP)
    # -------------------------
    def shap_global(self, X: Union[pd.DataFrame, np.ndarray]):
        Xdf = _to_df(X, feature_names=self.feature_names)
        Xs = Xdf.sample(min(len(Xdf), self.cfg.shap_sample), random_state=42)

        if hasattr(self.model, "predict_proba") and self.cfg.problem_type == "classification":
            f = lambda data: self.model.predict_proba(pd.DataFrame(data, columns=self.feature_names))
            explainer = shap.Explainer(f, Xs)
            shap_values = explainer(Xs)
            if shap_values.values.ndim == 3:
                shap.summary_plot(shap_values.values[:, :, 1], Xs, show=True)
            else:
                shap.summary_plot(shap_values, Xs, show=True)
        else:
            explainer = shap.Explainer(self.model.predict, Xs)
            shap_values = explainer(Xs)
            shap.summary_plot(shap_values, Xs, show=True)

        return shap_values

    def shap_local(self, X_row: Union[pd.DataFrame, np.ndarray]):
        Xdf = X_row if isinstance(X_row, pd.DataFrame) else pd.DataFrame([X_row], columns=self.feature_names)

        if hasattr(self.model, "predict_proba") and self.cfg.problem_type == "classification":
            f = lambda data: self.model.predict_proba(pd.DataFrame(data, columns=self.feature_names))
            explainer = shap.Explainer(f, Xdf)
            sv = explainer(Xdf)
            if sv.values.ndim == 3:
                shap.plots.waterfall(shap.Explanation(values=sv.values[0, :, 1],
                                                      base_values=sv.base_values[0, 1],
                                                      data=Xdf.iloc[0],
                                                      feature_names=self.feature_names))
            else:
                shap.plots.waterfall(sv[0])
        else:
            explainer = shap.Explainer(self.model.predict, Xdf)
            sv = explainer(Xdf)
            shap.plots.waterfall(sv[0])

        return sv

    def pdp(self, X: Union[pd.DataFrame, np.ndarray], features: List[Union[int, str]]):
        Xdf = _to_df(X, feature_names=self.feature_names)
        fig, ax = plt.subplots(figsize=(10, 5))
        PartialDependenceDisplay.from_estimator(self.model, Xdf, features=features, ax=ax)
        plt.title("Partial Dependence Plot(s)")
        plt.show()

    # -------------------------
    # 4) Counterfactuals (DiCE)
    # -------------------------
    def counterfactuals(self, query_instances: pd.DataFrame, total_CFs=3,
                        desired_class: str = "opposite", desired_range: Optional[List[float]] = None):
        if self.train_df is None:
            raise ValueError("train_df is required for counterfactuals. Pass it to fit().")

        data = dice_ml.Data(
            dataframe=self.train_df,
            continuous_features=[c for c in self.train_df.columns
                                 if c != self.target_name and pd.api.types.is_numeric_dtype(self.train_df[c])],
            outcome_name=self.target_name
        )

        if self.cfg.problem_type == "classification":
            m = dice_ml.Model(model=self.model, backend="sklearn", model_type="classifier")
        else:
            m = dice_ml.Model(model=self.model, backend="sklearn", model_type="regressor")

        dice = Dice(data, m, method="random")  # switch to "genetic" if you want
        cf = dice.generate_counterfactuals(
            query_instances=query_instances,
            total_CFs=total_CFs,
            desired_class=desired_class if self.cfg.problem_type == "classification" else None,
            desired_range=desired_range if self.cfg.problem_type == "regression" else None,
        )
        cf.visualize_as_dataframe()
        return cf

    # -------------------------
    # 5) Drift / PSI
    # -------------------------
    def drift_report(self, X_ref: Union[pd.DataFrame, np.ndarray], X_cur: Union[pd.DataFrame, np.ndarray], top_n=15) -> pd.DataFrame:
        Xr = _to_df(X_ref, feature_names=self.feature_names)
        Xc = _to_df(X_cur, feature_names=self.feature_names)

        rows = []
        for col in self.feature_names:
            if pd.api.types.is_numeric_dtype(Xr[col]):
                val = psi(Xr[col], Xc[col], buckets=min(self.cfg.bins, 10))
                rows.append({"feature": col, "psi": val})

        df = pd.DataFrame(rows).dropna().sort_values("psi", ascending=False)
        display_df = df.head(top_n)

        fig = px.bar(display_df, x="feature", y="psi", title=f"Top {top_n} Features by PSI (Higher = More Drift)")
        fig.show()

        for col in display_df["feature"].head(min(5, len(display_df))):
            tmp = pd.DataFrame({
                col: pd.concat([Xr[col].rename("ref"), Xc[col].rename("cur")], axis=0),
                "dataset": ["ref"] * len(Xr) + ["cur"] * len(Xc)
            })
            fig2 = px.histogram(tmp, x=col, color="dataset", barmode="overlay",
                                title=f"Distribution Drift: {col}")
            fig2.show()

        return df


/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# 1. Load a sample classification dataset
# X contains the features, y contains the labels
X, y = load_breast_cancer(return_X_y=True)

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Initialize the XGBoost classifier
# XGBoost provides a scikit-learn compatible estimator interface
model = XGBClassifier(objective='binary:logistic', use_label_encoder=False, eval_metric='logloss')

# 4. Fit the model to the training data
model.fit(X_train, y_train)

# 5. Make predictions on the test data
y_pred = model.predict(X_test)

# 6. Evaluate the model (optional)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy*100:.2f}%")

Accuracy: 95.61%


/Users/ajayparasa/.pyenv/versions/ModelXAI_p310/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [23:21:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [2]:
data = load_breast_cancer()

In [3]:
data

{'data': array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
         1.189e-01],
        [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
         8.902e-02],
        [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
         8.758e-02],
        ...,
        [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
         7.820e-02],
        [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
         1.240e-01],
        [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
         7.039e-02]], shape=(569, 30)),
 'target': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
        1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
        1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,

In [5]:
import pandas as pd
X_train_df = pd.DataFrame(X_train, columns=[f"f{i}" for i in range(X_train.shape[1])])

train_df = pd.concat(
    [X_train_df, pd.Series(y_train, name="target")],
    axis=1
)

X_test_df = pd.DataFrame(X_test, columns=X_train_df.columns)

In [6]:
# Example:
# model = ... (already trained sklearn classifier)
# X_train, y_train, X_test, y_test are your splits (X as DataFrame recommended)

cfg = FidxLiteConfig(problem_type="classification", shap_sample=1500, bins=10)
# fx = FidxLite(cfg).fit(model, X_train, y_train, train_df=pd.concat([X_train, pd.Series(y_train, name="target")], axis=1), target_name="target")
fx = FidxLite(cfg).fit(model, X_train_df, y_train, train_df=train_df, target_name="target")

# 1) Evaluation (metrics + ROC/PR/Calibration/KS/Gain)
out = fx.evaluate(X_test_df, y_test)

# 2) Threshold tuning (binary)
thr_df = fx.threshold_sweep(X_test_df, y_test)

# 3) Interpretability
fx.shap_global(X_test_df)
fx.shap_local(X_test_df.iloc[[0]])

# 4) Counterfactuals (binary)
query = X_test_df.iloc[[0]].copy()
fx.counterfactuals(query_instances=query, total_CFs=3, desired_class="opposite")

# 5) Drift (compare ref vs current)
fx.drift_report(X_ref=X_train_df, X_cur=X_test_df)


NameError: name 'FidxLiteConfig' is not defined